# JAK selectivity funnel — Stage A deep dive (Colab)

Loads a `loop_contract.json` exported from the dashboard (Stage B), **re-scores
through the same `src` models**, generates more-selective analogues, and emits a
before/after report — closing the funnel loop.

GPU is used here only (generation / optional docking); the deployed Stage-B app
stays CPU-only. Every generated molecule is an **in-silico hypothesis requiring
wet-lab validation** — never a hit.

In [ ]:
# Setup: clone the repo and install the pinned deps (Colab).
!git clone -q https://github.com/zqvo04/chem-predict-dashboard.git
%cd chem-predict-dashboard
!pip install -q -r requirements.txt

In [ ]:
# Upload the loop_contract.json exported by the dashboard's SELECT step.
from google.colab import files
up = files.upload()
contract_path = next(iter(up))

In [ ]:
# Run the deep dive. run_deep_dive asserts the contract's model_ids match the
# current Stage-B models, so re-scoring uses the IDENTICAL models (no divergence).
from src.deep_dive import run_from_file
result, a_contract, report = run_from_file(contract_path, max_analogues_per_case=30)
print(report)

### Optional GPU seams

- **Heavier generator:** replace `src.generate.generate_analogues` (a compact CPU
  decorator) with a GPU generative model (e.g. a fine-tuned scaffold-conditioned
  model) that emits SMILES; feed them to `funnel.score_molecules` exactly as below.
- **Confirmatory docking:** dock the shortlisted in-domain analogues into the JAK
  isoform structures (AutoDock Vina) as *orthogonal* evidence — consensus/pose
  only, never a second oracle. Attach the score to each molecule's `deep_dive` slot.

In [ ]:
# Before vs after: does generation shift the gap up while staying in-domain?
import matplotlib.pyplot as plt
b, a = result.before, result.after
plt.scatter([0]*len(b), b['gap'], c='tab:blue', label='before (selected)')
plt.scatter([1]*len(a), a['gap'], c=['#2ca02c' if d else '#bbb' for d in a['in_domain']])
plt.axhline(1.0, ls='--', color='grey'); plt.xticks([0,1], ['before','after'])
plt.ylabel('selectivity gap S'); plt.title('Loop closed: re-scored through the same models')
plt.show()

In [ ]:
# Save the A_rescore contract (feeds back into Stage B — the loop is a cycle).
from src.loop_contract import write_contract
write_contract(a_contract, 'loop_case_A_rescore.json')
print('wrote loop_case_A_rescore.json —', len(a_contract['molecules']), 'analogues')